# Base vs SFT vs DPO vs GRPO — Side-by-Side Comparison

Compares the untuned base model against the three trained adapters on held-out
eCFR questions. It reuses the cached generations in `results/generations_*.jsonl`
(produced by the `eval` stage), so it runs instantly without a GPU.
Run `python training_models_v1.py eval` first.

In [ ]:
import json
from pathlib import Path

from ecfr_pipeline import metrics

RESULTS = Path("results")
CHECKPOINTS = ["base", "sft", "dpo", "grpo"]

gens = {}
for name in CHECKPOINTS:
    p = RESULTS / f"generations_{name}.jsonl"
    if p.exists():
        gens[name] = {r["id"]: r for r in map(json.loads, p.read_text().splitlines()) if r}
print("loaded checkpoints:", list(gens), "| examples per checkpoint:",
      {k: len(v) for k, v in gens.items()})

In [ ]:
# Pick a diverse sample: one of each question type, preferring rows where the
# checkpoints disagree on citation correctness (the interesting cases).
common_ids = set.intersection(*(set(g) for g in gens.values()))

def cite_ok(rec):
    return metrics.citation_correct(rec["completion"], rec["expected_citation"])

by_type = {}
for ex_id in sorted(common_ids):
    row = gens[CHECKPOINTS[0]][ex_id]
    verdicts = {n: cite_ok(gens[n][ex_id]) for n in gens}
    disagree = len(set(verdicts.values())) > 1
    cur = by_type.get(row["type"])
    if cur is None or (disagree and not cur[1]):
        by_type[row["type"]] = (ex_id, disagree)

sample_ids = [v[0] for v in by_type.values()]
print("selected:", sample_ids)

In [ ]:
from IPython.display import Markdown, display

def show(ex_id):
    row = gens[CHECKPOINTS[0]][ex_id]
    parts = [
        f"### `{ex_id}`  ({row['type']})",
        f"**Question:** {row['prompt_messages'][1]['content']}",
        f"**Expected citation:** 12 CFR § {row['expected_citation']}",
        f"**Reference:** {row['reference'][:400]}{'…' if len(row['reference']) > 400 else ''}",
        "",
        "| checkpoint | cites correctly | token F1 | answer |",
        "|---|---|---|---|",
    ]
    for name in gens:
        rec = gens[name][ex_id]
        ok = "✅" if cite_ok(rec) else "❌"
        f1 = metrics.token_f1(rec["completion"], rec["reference"])
        ans = rec["completion"][:300].replace("\n", " ").replace("|", "\\|")
        parts.append(f"| **{name}** | {ok} | {f1:.3f} | {ans}{'…' if len(rec['completion']) > 300 else ''} |")
    display(Markdown("\n".join(parts)))

for ex_id in sample_ids:
    show(ex_id)

In [ ]:
# Aggregate view: overall metrics from the eval harness.
payload = json.loads((RESULTS / "eval_results.json").read_text())
rows = ["| checkpoint | citation_accuracy | wrong_citation_rate | token_f1 | rouge_l |",
        "|---|---|---|---|---|"]
for name, res in payload["checkpoints"].items():
    o = res["overall"]
    rows.append(f"| {name} | {o['citation_accuracy']:.4f} | {o['wrong_citation_rate']:.4f} "
                f"| {o['token_f1']:.4f} | {o['rouge_l']:.4f} |")
display(Markdown("\n".join(rows)))